# 🤖 AI Engineering Fundamentals — Lezione 4
## Notebook Gruppo B

**ITS Novitas 4.0 | Giovedì 28/05/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "B"
MEMBRI = ["", "", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
# ⚠️ Prima esecuzione: ChromaDB scarica Sentence Transformers (~90MB)
!pip install anthropic chromadb pypdf sentence-transformers -q

from google.colab import userdata
import anthropic, os, chromadb

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
chroma_client = chromadb.Client()

DOCUMENTO_WIDATA = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica
e qualità dell'aria (CO2, PM2.5). Classificazione IP67: impermeabile e resistente alla polvere.
Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni. Connettività: LoRaWAN, NB-IoT, WiFi.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare. Temperatura operativa: -40°C a +70°C.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato).

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
"""

print("✅ Setup completato!")

---
## 🎯 Tema del Gruppo B: Chunking & Embedding

Esplorate come la dimensione dei chunk e l'overlap
impattano la qualità del retrieval.
Trovate i parametri ottimali per il documento WiData.

---
### Esercizio 1 — Confronto chunk_size *(guidato)*

Stessa domanda, stesso documento, tre dimensioni di chunk diverse.
I chunk recuperati sono diversi? Quale dimensione trova le informazioni più utili?

In [ ]:
# Esercizio 1 — confronto chunk_size

def chunka_testo(testo, chunk_size=400, overlap=50):
    chunks = []
    start = 0
    while start < len(testo):
        chunk = testo[start:start+chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def crea_collection(nome, chunks):
    """Crea una collection ChromaDB con i chunk forniti."""
    try:
        chroma_client.delete_collection(nome)
    except:
        pass
    coll = chroma_client.create_collection(nome)
    coll.add(documents=chunks, ids=[str(i) for i in range(len(chunks))])
    return coll

def cerca_in_collection(domanda, collection, n=2):
    risultati = collection.query(query_texts=[domanda], n_results=n)
    return risultati["documents"][0]

# Crea tre collection con dimensioni diverse
dimensioni = [100, 400, 800]
collections = {}

for dim in dimensioni:
    chunks = chunka_testo(DOCUMENTO_WIDATA, chunk_size=dim, overlap=20)
    collections[dim] = crea_collection(f"widata_b_{dim}", chunks)
    print(f"chunk_size={dim}: {len(chunks)} chunk generati")

print()

# Stessa domanda sulle tre collection
domanda = "Qual è l'autonomia della batteria del sensore XS200?"
print(f"❓ {domanda}\n")

for dim in dimensioni:
    print(f"{'='*50}")
    print(f"chunk_size = {dim}")
    # TODO: cercate nella collection corrispondente
    chunks_trovati = cerca_in_collection(domanda, ___)
    for i, chunk in enumerate(chunks_trovati):
        print(f"Chunk {i+1}: {chunk[:150]}...")
    print()

# Quale chunk_size trova le informazioni più utili? Perché?
# ...

---
### Esercizio 2 — L'effetto dell'overlap *(guidato)*

Spezzate lo stesso testo con overlap=0 e overlap=100.
Trovate un caso in cui la frase chiave è a cavallo tra due chunk
e verificate che l'overlap la recuperi correttamente.

In [ ]:
# Esercizio 2 — effetto dell'overlap

# Testo di test dove l'informazione chiave è a cavallo tra due chunk
testo_test = """Il sensore XS200 è resistente alle intemperie grazie alla classificazione
IP67 che garantisce impermeabilità completa e protezione dalla polvere.
La connettività LoRaWAN permette trasmissioni fino a 15km in campo aperto."""

# Chunk size piccolo per forzare lo spezzamento nel punto critico
CHUNK_SIZE = 80

# SENZA overlap
chunks_no_overlap = chunka_testo(testo_test, chunk_size=CHUNK_SIZE, overlap=0)
print("SENZA overlap:")
for i, c in enumerate(chunks_no_overlap):
    print(f"  Chunk {i+1}: '{c}'")

print()

# CON overlap
chunks_overlap = chunka_testo(testo_test, chunk_size=CHUNK_SIZE, overlap=30)
print("CON overlap=30:")
for i, c in enumerate(chunks_overlap):
    print(f"  Chunk {i+1}: '{c}'")

print()

# TODO: create due collection e cercate 'impermeabilità LoRaWAN'
# Quale versione recupera meglio le informazioni?
coll_no = crea_collection("test_no_overlap", chunks_no_overlap)
coll_si = crea_collection("test_overlap", chunks_overlap)

domanda_test = "Il sensore è impermeabile e come trasmette i dati?"

print(f"❓ {domanda_test}")
print("\nSENZA overlap — chunk recuperato:")
# TODO: cercate in coll_no
print(___)
print("\nCON overlap — chunk recuperato:")
# TODO: cercate in coll_si
print(___)

# L'overlap fa la differenza in questo caso? Quando è più importante?
# ...

---
### Esercizio 3 — Trovare i parametri ottimali *(libero)*

Create un mini-benchmark: 5 domande sul documento WiData
con risposte attese note. Testate almeno 4 combinazioni
di chunk_size e overlap. Quale combinazione risponde
correttamente al maggior numero di domande?

In [ ]:
# Esercizio 3 — benchmark parametri chunking

# Dataset di test: domanda + risposta attesa (parola chiave)
dataset = [
    {"domanda": "Qual è l'autonomia del sensore XS200?", "atteso": "2 anni"},
    {"domanda": "Quanti sensori gestisce il gateway GW500?", "atteso": "1000"},
    {"domanda": "Quanto costa il piano Pro di Xplore?", "atteso": "49"},
    {"domanda": "Qual è il numero di telefono del supporto?", "atteso": "079"},
    {"domanda": "Qual è la classificazione IP del sensore?", "atteso": "IP67"},
]

# Combinazioni da testare
configurazioni = [
    {"chunk_size": 100, "overlap": 10},
    {"chunk_size": 200, "overlap": 30},
    {"chunk_size": 400, "overlap": 50},
    {"chunk_size": 800, "overlap": 100},
]

print(f"{'Config':<25} {'Corrette/5':<15} {'Score'}")
print("-" * 50)

for conf in configurazioni:
    cs = conf["chunk_size"]
    ov = conf["overlap"]

    # TODO: create la collection con questa configurazione
    # Per ogni domanda nel dataset:
    #   - cercate i chunk rilevanti
    #   - verificate se la parola attesa è nei chunk recuperati
    # Constate quante domande trovano la risposta corretta

    corrette = 0
    # ...

    print(f"size={cs}, overlap={ov:<10} {corrette}/5")

print()
print("Configurazione ottimale per WiData: chunk_size=___, overlap=___")
print("Motivazione: ...")

---
### Esercizio 4 — Chunking semantico *(libero)*

Il chunking a lunghezza fissa è semplice ma spezza le frasi.
Implementate una versione che spezza sui paragrafi naturali
del documento. Confrontate con il chunking a lunghezza fissa.

In [ ]:
# Esercizio 4 — chunking semantico sui paragrafi

def chunka_per_paragrafi(testo, max_chunk_size=600):
    """
    Spezza il testo sui paragrafi (doppio newline).
    Se un paragrafo è troppo lungo, lo spezza ulteriormente.
    """
    # TODO: dividete il testo per \n\n
    # Per ogni paragrafo: se è più lungo di max_chunk_size, spezzatelo
    # altrimenti usatelo direttamente
    # Filtrate i chunk vuoti
    ___

chunks_semantici = chunka_per_paragrafi(DOCUMENTO_WIDATA)
chunks_fissi = chunka_testo(DOCUMENTO_WIDATA, chunk_size=400, overlap=50)

print(f"Chunking fisso:    {len(chunks_fissi)} chunk")
print(f"Chunking semantico: {len(chunks_semantici)} chunk")
print()

# Mostrate i chunk semantici
print("Chunk semantici:")
for i, c in enumerate(chunks_semantici):
    print(f"  Chunk {i+1} ({len(c)} char): {c[:80]}...")

print()

# TODO: create due collection e confrontate la qualità del retrieval
# con le stesse 5 domande del benchmark
# Il chunking semantico è migliore? In quali casi?
# ...

---
## 📊 Preparate la presentazione (5 slide)

1. **Chunk troppo piccoli vs ottimali vs grandi** — con i vostri esempi concreti
2. **L'effetto dell'overlap** — mostrate il caso della frase spezzata
3. **Il vostro benchmark** — tabella con i risultati delle 4 configurazioni
4. **Chunking semantico vs fisso** — differenze e quando usare quale
5. **La vostra raccomandazione** — parametri ottimali per WiData con motivazione

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*